<a href="https://colab.research.google.com/github/Kanika-0905/kanika-codeboosters-2026/blob/main/Day_07_Semantic_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install chromadb sentence-transformers -q
print("Installation completed!!!")
#384 dimensional number

Installation completed!!!


In [12]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
import chromadb


print(f'Chromadb version: {chromadb.__version__}')

Chromadb version: 1.5.9


In [13]:
documents = [
    "ETL is used to clean and transform data",
    "A vehicle is a mode of transportation",
    "Cars and trucks are popular automobile",
    "SQL is used to query databases",
    "Machine learning trains models on data",
]

query_keyboard = "vehicle"
print("="*30)
print(f"Keyword search for: {query_keyboard}")
print("="*30)

for i, doc in enumerate(documents):
  if query_keyboard.lower() in doc.lower():
    print(f" FOUND [doc_{i}]:{doc}")
  else:
    print(f" MISSED [doc_{i}]:{doc}")
print()


Keyword search for: vehicle
 MISSED [doc_0]:ETL is used to clean and transform data
 FOUND [doc_1]:A vehicle is a mode of transportation
 MISSED [doc_2]:Cars and trucks are popular automobile
 MISSED [doc_3]:SQL is used to query databases
 MISSED [doc_4]:Machine learning trains models on data



In [18]:
print("Loading embedding model")
model=SentenceTransformer('all-MiniLM-L6-v2')
print(f"Model products vectors of size:{model.get_embedding_dimension()}")


Loading embedding model


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model products vectors of size:384


In [22]:
sentence = "ETL is used to clean and transform data"
embedding = model.encode(sentence)
print(f"Input sentence: {sentence}")
print()
print(f"Embedding type: {type(embedding)}")

print(f'Embeding shape:{embedding.shape}')

print(f"First 10 numbers:{embedding[:10].round(4)}")

print(f"Min value : {embedding.min():.4f}")
print(f"Max value: {embedding.max():.4f}")

Input sentence: ETL is used to clean and transform data

Embedding type: <class 'numpy.ndarray'>
Embeding shape:(384,)
First 10 numbers:[-0.0784  0.0541  0.0224 -0.0389  0.0221 -0.0904  0.0007 -0.0152  0.0733
  0.0362]
Min value : -0.1381
Max value: 0.1815


In [33]:
documents = [
    "ETL is used to clean and transform data",
    "A vehicle is a mode of transportation",
    "Cars and trucks are popular automobile",
    "SQL is used to query databases",
    "Machine learning trains models on data",
]
embedding=model.encode(documents)
query_keyboard = "vehicle"
key_encode = model.encode(query_keyboard)
print("="*30)
print(f"Keyword search for: {query_keyboard}")
print("="*30)

for i, doc in enumerate(embedding):
  if query_keyboard.lower() in doc.lower():
    print(f" FOUND [doc_{i}]:{doc}")
  else:
    print(f" MISSED [doc_{i}]:{doc}")
print()

Keyword search for: vehicle


AttributeError: 'numpy.ndarray' object has no attribute 'lower'

In [41]:
from sklearn.metrics.pairwise import cosine_similarity

sentences=["Re-election date are going to be announced",
           "Laptop price are high due to manufacture",
           "Politicians are massively exchange their parties",
           ]

embeddings = model.encode(sentences)
sim_your_01=cosine_similarity([embeddings[0]],[embeddings[1]])
sim_your_02=cosine_similarity([embeddings[0]],[embeddings[2]])
print(f"Number of sentences: {len(sentences)}")
print(f"Embeddings shape: {embeddings.shape}")
print()
print("Each row is one sentence's embedding:")
print(f"Similarity(A vs B):{sim_your_01[0][0]:.4f}")
print(f"Similarity(A vs C):{sim_your_02[0][0]:.4f}")

Number of sentences: 3
Embeddings shape: (3, 384)

Each row is one sentence's embedding:
Similarity(A vs B):-0.0063
Similarity(A vs C):0.2227


In [45]:
chroma_client=chromadb.Client()
collection=chroma_client.get_or_create_collection("demo_table")
print(f"Document count: {collection.count()}")

Document count: 0


In [47]:
sample_docs=[
    "ETL is used to clean and transform data",
    "A vehicle is a mode of transportation",
    "Cars and trucks are popular automobile",
    "SQL is used to query databases",
    "Machine learning trains models on data",
]

sample_ids=["doc001","doc002","doc003","doc004","doc005"]
sample_metadata=[
    {"subject":"Data engineering","topic":"ETL"},
    {"subject":"Transaportation","topic":"vehicle"},
    {"subject":"Transaportation","topic":"automobile"},
    {"subject":"Data engineering","topic":"SQL"},
    {"subject":"Data science","topic":"ML"},
]

collection.add(
    documents=sample_docs,
    ids=sample_ids,
    metadatas=sample_metadata
)
print("Collection count",collection.count())

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 68.7MiB/s]


Collection count 5


In [53]:
query = "How do i clean and prepare data?"
print(f"Query: '{query}'")
print("=" *60)
print()

# Encode the query into an embedding using the same model
query_embedding = model.encode([query]).tolist()

# Perform a similarity search in the ChromaDB collection
# The 'collection' and 'model' variables are expected to be defined in previous cells.
results = collection.query(
    query_embeddings=query_embedding,
    n_results=3
)

matched_docs = results['documents'][0]
matched_ids = results['ids'][0]
matched_metadata = results['metadatas'][0]
matched_distances = results['distances'][0]


for rank, (doc, id, meta, dist) in enumerate(zip(matched_docs, matched_ids, matched_metadata, matched_distances)):
  print(f"Rank {rank} | ID:{doc,id} | Distance:{dist:.4f}")
  print(f"Subject: {meta['subject']} | Topic: {meta['topic']}")
  print(f"Document: {doc}")
  print()

  print("NOTICE: The results are about ETL ad Pandas - exactly what 'clean and prepare data")
  print("Sematic search found them even though the words are different")

Query: 'How do i clean and prepare data?'

Rank 0 | ID:('ETL is used to clean and transform data', 'doc001') | Distance:1.1972
Subject: Data engineering | Topic: ETL
Document: ETL is used to clean and transform data

NOTICE: The results are about ETL ad Pandas - exactly what 'clean and prepare data
Sematic search found them even though the words are different
Rank 1 | ID:('SQL is used to query databases', 'doc004') | Distance:1.5369
Subject: Data engineering | Topic: SQL
Document: SQL is used to query databases

NOTICE: The results are about ETL ad Pandas - exactly what 'clean and prepare data
Sematic search found them even though the words are different
Rank 2 | ID:('Machine learning trains models on data', 'doc005') | Distance:1.7309
Subject: Data science | Topic: ML
Document: Machine learning trains models on data

NOTICE: The results are about ETL ad Pandas - exactly what 'clean and prepare data
Sematic search found them even though the words are different


In [ ]:
filtered_results=collection.query(
    query_texts=[filtered_query],
    n_results=3,
    where={"subject":"Machine Learning"}
)
print(f"Filtered query: {filtered_query}")
print("Filter : Only machine learning documnets")
print("="*60)
for rank,

In [56]:
print("Distance to similarity conversion")
print("="*50)
print(f"{'distance':<15}{'Similarity':<15}{'Interpretation':<20}")
print("="*50)

distance = [0.05,0.02,0.40,0.05,0.09]
inter=["Near identical","Very similar","Related","Somewhat related","Not"]

for dist,interp in zip(distance,inter):
  similarity=1-dist
  print(f"{dist:<15}{similarity:<15.4f}{interp:<20}")

Distance to similarity conversion
distance       Similarity     Interpretation      
0.05           0.9500         Near identical      
0.02           0.9800         Very similar        
0.4            0.6000         Related             
0.05           0.9500         Somewhat related    
0.09           0.9100         Not                 
